In [10]:
%pip install pandas numpy scikit-learn imbalanced-learn xgboost

In [11]:
import pandas as pd
import numpy as np

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Handling imbalance
from imblearn.over_sampling import SMOTE

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [24]:
df = pd.read_csv("cumulative.csv")

print(df.head())
print(df.shape)

   rowid     kepid kepoi_name   kepler_name koi_disposition koi_pdisposition  \
0      1  10797460  K00752.01  Kepler-227 b       CONFIRMED        CANDIDATE   
1      2  10797460  K00752.02  Kepler-227 c       CONFIRMED        CANDIDATE   
2      3  10811496  K00753.01           NaN  FALSE POSITIVE   FALSE POSITIVE   
3      4  10848459  K00754.01           NaN  FALSE POSITIVE   FALSE POSITIVE   
4      5  10854555  K00755.01  Kepler-664 b       CONFIRMED        CANDIDATE   

   koi_score  koi_fpflag_nt  koi_fpflag_ss  koi_fpflag_co  ...  \
0      1.000              0              0              0  ...   
1      0.969              0              0              0  ...   
2      0.000              0              1              0  ...   
3      0.000              0              1              0  ...   
4      1.000              0              0              0  ...   

   koi_steff_err2  koi_slogg  koi_slogg_err1  koi_slogg_err2  koi_srad  \
0           -81.0      4.467           0.064    

In [13]:
print(df.isnull().sum())

rowid                   0
kepid                   0
kepoi_name              0
kepler_name          7270
koi_disposition         0
koi_pdisposition        0
koi_score            1510
koi_fpflag_nt           0
koi_fpflag_ss           0
koi_fpflag_co           0
koi_fpflag_ec           0
koi_period              0
koi_period_err1       454
koi_period_err2       454
koi_time0bk             0
koi_time0bk_err1      454
koi_time0bk_err2      454
koi_impact            363
koi_impact_err1       454
koi_impact_err2       454
koi_duration            0
koi_duration_err1     454
koi_duration_err2     454
koi_depth             363
koi_depth_err1        454
koi_depth_err2        454
koi_prad              363
koi_prad_err1         363
koi_prad_err2         363
koi_teq               363
koi_teq_err1         9564
koi_teq_err2         9564
koi_insol             321
koi_insol_err1        321
koi_insol_err2        321
koi_model_snr         363
koi_tce_plnt_num      346
koi_tce_delivname     346
koi_steff   

In [14]:
df = df.fillna(df.mean(numeric_only=True))

In [15]:
drop_columns = ['kepid', 'kepoi_name']

for col in drop_columns:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

In [16]:
print(df['koi_disposition'].value_counts())

koi_disposition
FALSE POSITIVE    5023
CONFIRMED         2293
CANDIDATE         2248
Name: count, dtype: int64


In [17]:
df['koi_disposition'] = df['koi_disposition'].map({
    'CONFIRMED': 1,
    'FALSE POSITIVE': 0,
    'CANDIDATE': 0
})

In [18]:
X = df.drop('koi_disposition', axis=1)
y = df['koi_disposition']

In [19]:
X = pd.get_dummies(X)

In [20]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [21]:
import piplite
await piplite.install(['imbalanced-learn', 'xgboost'])

In [28]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Drop any columns that are entirely empty (like koi_teq_err1 and koi_teq_err2)
X_clean = X.dropna(axis=1, how='all')
y_clean = y

# 2. Scale the data safely now that all NaNs are gone
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

# 3. Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

# 4. Apply SMOTE seamlessly
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 5. Verify results
print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_smote).value_counts().to_dict())

/lib/python3.13/site-packages/threadpoolctl.py:1123: RuntimeWarning: JsProxy.as_object_map() is deprecated. Use as_py_json() instead.
  for filepath in LDSO.loadedLibsByName.as_object_map():


Before SMOTE: {0: 5817, 1: 1834}
After SMOTE: {0: 5817, 1: 5817}


In [29]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [30]:
xgb_model = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_smote, y_train_smote)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [33]:
nn_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=300,
    random_state=42
)

nn_model.fit(X_train_smote, y_train_smote)

,hidden_layer_sizes,"(100, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,300
,shuffle,True
,random_state,42


In [34]:
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)
nn_pred = nn_model.predict(X_test)

In [35]:
def evaluate_model(name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n{name} Performance")
    print("-" * 30)

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

In [36]:
evaluate_model("Random Forest", y_test, rf_pred)

evaluate_model("XGBoost", y_test, xgb_pred)

evaluate_model("Neural Network", y_test, nn_pred)


Random Forest Performance
------------------------------
Accuracy : 0.9143
Precision: 0.7875
Recall   : 0.8802
F1 Score : 0.8313

XGBoost Performance
------------------------------
Accuracy : 0.919
Precision: 0.798
Recall   : 0.8867
F1 Score : 0.84

Neural Network Performance
------------------------------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


In [39]:
results = []

models = {
    "Random Forest": rf_pred,
    "XGBoost": xgb_pred,
    "Neural Network": nn_pred
}

for name, pred in models.items():

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)

print(results_df)

            Model  Accuracy  Precision    Recall  F1 Score
0   Random Forest  0.914271   0.787524  0.880174  0.831276
1         XGBoost  0.918975   0.798039  0.886710  0.840041
2  Neural Network  1.000000   1.000000  1.000000  1.000000


In [41]:
import pickle
from IPython.display import display, HTML
import base64

# 1. Save the trained XGBoost model and the scaler into a dictionary
model_data = {
    "model": xgb_model,   
    "scaler": scaler
}

# 2. Serialize the data using pickle
pickle_data = pickle.dumps(model_data)

# 3. Create a direct browser download link
b64 = base64.b64encode(pickle_data).decode()
href = f'<a href="data:application/octet-stream;base64,{b64}" download="kepler_xgb_model.pkl" style="padding: 10px; background-color: #4CAF50; color: white; text-decoration: none; border-radius: 5px;">📥 Click Here to Download Your Model File</a>'

# 4. Display the button
display(HTML(href))